In [2]:
#say no to warnings!
import warnings
warnings.filterwarnings("ignore")
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = '3'
import tensorflow as tf
tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)

In [ ]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.backend import clear_session
from tensorflow.keras.layers import Embedding, Dense, TimeDistributed, Bidirectional, LSTM
import pandas as pd
import numpy as np

RANDOM_STATE = 1

# Machine Translation with RNNs

Let's tackle a machine translation task using Recurrent Neural Networks (RNNs).

## Dataset

The dataset consists of 50,000 Italian-English sentence pairs, provided as part of the course material. 

Each entry contains a sentence in Italian alongside its corresponding English translation, making it a parallel corpus suited for sequence-to-sequence learning.

In [7]:
# read the dataset
df = pd.read_csv("data/machine_translation.csv")

df.head

<bound method NDFrame.head of                                 italian               english
0                     tom portò i suoi.      tom brought his.
1              a te non piace il pesce?  don't you like fish?
2                 non abbiamo mai riso.     we never laughed.
3                   aspetti un momento.     hang on a moment.
4                      quando è finito?    when did that end?
...                                 ...                   ...
49995          non la riesco a salvare.     i can't save you.
49996                  lui è scomparso.       he disappeared.
49997             ha più di trent'anni.     he's over thirty.
49998            il cane sta ansimando.   the dog is panting.
49999  spero di non essere il prossimo.  i hope i'm not next.

[50000 rows x 2 columns]>

In [8]:
eng_senteces = df.english
ita_sentences = df.italian

X_train, X_test, y_train, y_test = train_test_split(eng_senteces, ita_sentences, test_size=.2, random_state=RANDOM_STATE)

## Preprocessing

In [11]:
eng_tokenizer = Tokenizer()
eng_tokenizer.fit_on_texts(X_train)

eng_sequences_train = eng_tokenizer.texts_to_sequences(X_train)
eng_sequences_test = eng_tokenizer.texts_to_sequences(X_test)

max_eng_len = len(max(eng_sequences_train, key=len))

padded_eng_sequences_train = pad_sequences(eng_sequences_train, maxlen=max_eng_len)
padded_eng_sequences_test = pad_sequences(eng_sequences_test, maxlen=max_eng_len)

eng_vocabulary_size = len(eng_tokenizer.word_index)+1


ita_tokenizer = Tokenizer()
ita_tokenizer.fit_on_texts(y_train)

ita_sequences_train = ita_tokenizer.texts_to_sequences(y_train)
ita_sequences_test = ita_tokenizer.texts_to_sequences(y_test)

max_ita_len = len(max(ita_sequences_train, key=len))

padded_ita_sequences_train = pad_sequences(ita_sequences_train, maxlen=max_ita_len)
padded_ita_sequences_test = pad_sequences(ita_sequences_test , maxlen=max_ita_len)

ita_vocabulary_size = len(ita_tokenizer.word_index)+1

In [12]:
max_eng_len, max_ita_len

(6, 10)

In [16]:
X_temp = pad_sequences(padded_eng_sequences_train, max_ita_len)

## Model

In [25]:
clear_session()
model = Sequential()
model.add(Embedding(input_dim=eng_vocabulary_size, output_dim=128, input_length=max_ita_len))
model.add(Bidirectional(LSTM(64, return_sequences=True, activation='tanh')))
model.add(Bidirectional(LSTM(64, return_sequences=True, activation='tanh')))
model.add(TimeDistributed(Dense(ita_vocabulary_size, activation='softmax')))

model.compile(optimizer='rmsprop', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

model.fit(X_temp, padded_ita_sequences_train, batch_size=512, validation_split=.2, epochs=180)

Epoch 1/180
63/63 ━━━━━━━━━━━━━━━━━━━━ 40s 578ms/step - accuracy: 0.6416 - loss: 4.7439 - val_accuracy: 0.6536 - val_loss: 2.7335
Epoch 2/180
63/63 ━━━━━━━━━━━━━━━━━━━━ 35s 554ms/step - accuracy: 0.6522 - loss: 2.5402 - val_accuracy: 0.6536 - val_loss: 2.4359
Epoch 3/180
63/63 ━━━━━━━━━━━━━━━━━━━━ 39s 616ms/step - accuracy: 0.6575 - loss: 2.4007 - val_accuracy: 0.6628 - val_loss: 2.3714
Epoch 4/180
63/63 ━━━━━━━━━━━━━━━━━━━━ 40s 626ms/step - accuracy: 0.6643 - loss: 2.3498 - val_accuracy: 0.6671 - val_loss: 2.3318
Epoch 5/180
63/63 ━━━━━━━━━━━━━━━━━━━━ 39s 618ms/step - accuracy: 0.6663 - loss: 2.3138 - val_accuracy: 0.6681 - val_loss: 2.3027
Epoch 6/180
63/63 ━━━━━━━━━━━━━━━━━━━━ 41s 643ms/step - accuracy: 0.6681 - loss: 2.2825 - val_accuracy: 0.6703 - val_loss: 2.2754
Epoch 7/180
63/63 ━━━━━━━━━━━━━━━━━━━━ 40s 628ms/step - accuracy: 0.6696 - loss: 2.2537 - val_accuracy: 0.6706 - val_loss: 2.2524
Epoch 8/180
63/63 ━━━━━━━━━━━━━━━━━━━━ 40s 634ms/step - accuracy: 0.6710 - loss: 2.2269 - 

## Evaluation

In [26]:
test_X = pad_sequences(padded_eng_sequences_test, maxlen=max_ita_len)
model.evaluate(test_X, padded_ita_sequences_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 9s 29ms/step - accuracy: 0.7845 - loss: 1.1321


[1.1320639848709106, 0.7845001816749573]

## Visualization

In [29]:
def logits_to_txt(logits, tokenizer):
    index_to_words = {id: word for word, id in tokenizer.word_index.items()}
    index_to_words[0] = '<PAD>'
    return ' '.join([index_to_words[prediction] for prediction in np.argmax(logits, 1)])

In [ ]:
emb_preds = model.predict(test_X)

In [38]:
emb_preds

array([[[9.99949574e-01, 2.18170044e-06, 3.57103227e-06, ...,
         2.58234354e-11, 2.59659256e-11, 2.53175415e-11],
        [9.99994040e-01, 4.82537189e-07, 3.52783388e-07, ...,
         1.33705254e-12, 1.40053767e-12, 1.34941938e-12],
        [9.99991179e-01, 5.58970385e-07, 1.03731441e-06, ...,
         2.48424263e-12, 2.64902596e-12, 2.49350410e-12],
        ...,
        [5.22327609e-04, 9.98745859e-01, 3.22026026e-04, ...,
         1.36561942e-10, 1.26178040e-10, 1.17309079e-10],
        [4.31037588e-05, 7.46680671e-05, 9.91817296e-01, ...,
         6.33351260e-10, 6.66237454e-10, 5.94000960e-10],
        [8.98518949e-04, 8.82558743e-05, 5.58060885e-04, ...,
         1.48729453e-06, 1.65755841e-06, 1.40434156e-06]],

       [[9.99986529e-01, 1.30482704e-07, 1.59501312e-07, ...,
         1.68111347e-12, 1.50393402e-12, 1.60293382e-12],
        [9.99998808e-01, 1.61901461e-08, 3.77843357e-09, ...,
         4.83843244e-14, 4.38324033e-14, 4.61749881e-14],
        [9.99998689e-01, 

In [42]:
for i in range(0, 20):
    print(X_test.iloc[i])
    print(logits_to_txt(emb_preds[i], ita_tokenizer))
    print('--------------------------------------\n')

tom is awake.
<PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> tom è sveglio
--------------------------------------

i just did it.
<PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> l'ho già già
--------------------------------------

tom is brilliant.
<PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> tom è brillante
--------------------------------------

allow me to help.
<PAD> <PAD> <PAD> <PAD> <PAD> <PAD> mi mi ad aiutare
--------------------------------------

we heard tom.
<PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> abbiamo sentito tom
--------------------------------------

don't waste my time.
<PAD> <PAD> <PAD> <PAD> <PAD> non è del mio tempo
--------------------------------------

you were sick.
<PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> eri malato
--------------------------------------

tom plays piano.
<PAD> <PAD> <PAD> <PAD> <PAD> <PAD> tom gioca il piano
--------------------------------------

i'm resting now.
<PAD> <PAD> <PAD> <PAD> <PAD> <PAD> sto sto paura ora
-----------------------------